# LesioScan — Exploratory Data Analysis (EDA)
Notebook Google Colab. Jalankan setelah PAD-UFES-20 sudah diekstrak ke `data/images` dan metadata ke `data/metadata.csv`. Analisis menyeluruh adalah deskriptif; keputusan model dan preprocessing hanya boleh menggunakan **train set** setelah split.

In [ ]:
%cd /content/lesioscan-ml
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image
ROOT=Path('/content/lesioscan-ml')
DATA=ROOT/'data'
df=pd.read_csv(DATA/'metadata.csv')
print('Rows',len(df),'Unique patients',df['patient_id'].nunique(),'Unique images',df['img_id'].nunique())
display(df.head())

## 1. Pemeriksaan kelengkapan data

In [ ]:
display(df.isna().sum().to_frame('nilai_kosong'))
print('Duplikasi img_id:',df['img_id'].duplicated().sum())
missing=df[~df['img_id'].astype(str).map(lambda n:(DATA/'images'/n).is_file())]
print('File gambar hilang:',len(missing));display(missing.head())

## 2. Distribusi kelas — deskriptif

In [ ]:
counts=df['diagnostic'].value_counts().sort_index()
display(counts.to_frame('gambar'))
ax=counts.plot.bar(figsize=(8,4),title='Distribusi Kelas')
ax.set(xlabel='Kelas',ylabel='Jumlah gambar');plt.tight_layout();plt.show()

## 3. Gambar per pasien

In [ ]:
patient_counts=df.groupby('patient_id').size()
display(patient_counts.describe().to_frame('gambar_per_pasien'))
patient_counts.hist(bins=25);plt.title('Jumlah gambar per pasien');plt.xlabel('Gambar');plt.ylabel('Jumlah pasien');plt.show()

## 4. Sampel gambar tiap kelas
**Gunakan hanya untuk eksplorasi visual, bukan membuat diagnosis atau mengklaim akurasi.**

In [ ]:
from IPython.display import display
for cls in sorted(df['diagnostic'].dropna().unique()):
    row=df[df['diagnostic']==cls].iloc[0]
    path=DATA/'images'/str(row['img_id'])
    if path.is_file():
        print('Kelas:',cls,'File:',path.name)
        with Image.open(path) as im:display(im.convert('RGB').resize((224,224)))

## 5. EDA otomatis dan laporan
Menjalankan script membuat CSV/grafik di `outputs/eda`.

In [ ]:
!python scripts/eda.py --out outputs/eda
print('EDA outputs:',[p.name for p in Path('outputs/eda').iterdir()])

## 6. Setelah split: audit kebocoran pasien

In [ ]:
split_file=DATA/'split.csv'
if split_file.exists():
    split=pd.read_csv(split_file)
    display(pd.crosstab(split['split'],split['diagnostic']))
    sets={s:set(split.loc[split['split']==s,'patient_id']) for s in ('train','val','test')}
    for a,b in [('train','val'),('train','test'),('val','test')]:
        print(f'{a} vs {b}:',len(sets[a]&sets[b]),'pasien sama')
else:
    print('Belum ada data/split.csv. Jalankan scripts/prepare.py setelah EDA awal.')

## 7. Kesimpulan EDA — isi setelah ada hasil
- Distribusi kelas: ...
- Kelengkapan data: ...
- Karakteristik pasien/gambar: ...
- Dampak terhadap training set: ...
- Keterbatasan: ...